# 03 — Gold | Indicadores da CAT — Paraná

**Fonte:** `laest.silver.cat_2025`
**Abrangência geográfica:** Paraná — acidente **ocorrido** no PR (`uf_acidente = '41'`), independentemente do município de notificação ou residência

## Universos das análises (não misturados)

1. **Perfil dos acidentes de trabalho** — *todos* os acidentes de trabalho ocorridos no PR → `laest.gold.cat_perfil_pr` (formato longo: dimensão × categoria, com n e %; % calculado dentro de cada dimensão)
2. **Acidentes de trabalho no trânsito** — acidentes ocorridos no PR com **causa CID-10 V01–V99** (acidentes de transporte) → `laest.gold.cat_transito_pr_ano` (n por ano + % motocicleta) e `laest.gold.cat_transito_pr_causas` (n e % por causa/ano, base para a evolução anual)
3. **Lesões em acidentes de transporte** — mesmo universo do trânsito → `laest.gold.cat_transito_pr_lesoes` (top 10 lesões com código CID-10, descrição, n e %)

## Filtros e regras documentadas

* Trânsito = `causa_acidente_cid10` iniciando em **V** (V01–V99), após **normalização** que remove ponto final (ex.: `'V23.'` → `'V23'`)
* Motocicleta = causas **V20–V29** (motociclista traumatizado em acidente de transporte)
* Faixa etária derivada de `idade_anos` (menos de 18, 18-24, 25-34, 35-44, 45-54, 55-64, 65+); idade ausente → `Ignorado`
* Registros ignorados/em branco **preservados** como categorias: CBO `XXX`, situação de trabalho `99`, nulos → `Ignorado`
* % sempre sobre o total do universo correspondente: por dimensão (análise 1), por ano (análise 2) e sobre o total de acidentes de transporte (análise 3)
* Classificação por **códigos CBO e CID-10**; descrições CID-10 adicionadas como metadado de referência (código não mapeado fica sem descrição)

## ⚠️ Períodos solicitados × períodos disponíveis

Os períodos pedidos (2024; 2016–2025; 2015–2025) **não estão na base atual**: o arquivo ingestado (`ACGRBR25.dbc`) cobre **apenas 2025** — nenhum valor é inserido manualmente. Para os demais anos: baixar `ACGRBRaa.dbc` no SINAN, subir em `laest.bronze.landing_raw` e rodar os notebooks `01_bronze` + `02_silver`; as consultas desta camada usam a coluna `ano` e absorvem os novos períodos automaticamente.

**Autor/Data:** Vitor Silva — 04/11/2026

In [0]:
# RESUMO — dados disponíveis para os 3 universos de análise
# Rodar sempre primeiro: mostra os períodos e o tamanho de cada universo
from pyspark.sql import functions as F

SILVER = 'laest.silver.cat_2025'
t = spark.table(SILVER)

print('=== Anos cobertos pela base atual ===')
t.groupBy('ano').agg(F.count('*').alias('registros')).orderBy('ano').show()

pr = t.filter(F.col('uf_acidente') == '41')
print(f'=== Acidentes de trabalho ocorridos no PR: {pr.count():,} ===')

tr = (pr.withColumn('causa', F.regexp_replace(F.col('causa_acidente_cid10'), r'\.$', ''))
        .filter(F.col('causa').rlike('^V')))
print(f'=== Desses, acidentes de transporte (CID-10 V01-V99): {tr.count():,} ===')

print('=== Cobertura das dimensões do perfil no PR (% preenchido) ===')
pr.select(
    F.round(100 * F.count_if(F.col('ocupacao_cbo').isNotNull()) / F.count('*'), 1).alias('pct_ocupacao_cbo'),
    F.round(100 * F.count_if(F.col('sexo').isNotNull()) / F.count('*'), 1).alias('pct_sexo'),
    F.round(100 * F.count_if(F.col('raca_cor').isNotNull()) / F.count('*'), 1).alias('pct_raca_cor'),
    F.round(100 * F.count_if(F.col('idade_anos').isNotNull()) / F.count('*'), 1).alias('pct_faixa_etaria'),
    F.round(100 * F.count_if(F.col('situacao_trabalho_cod').isNotNull()) / F.count('*'), 1).alias('pct_situacao_mercado'),
    F.round(100 * F.count_if(F.col('cat_emitida_cod').isNotNull()) / F.count('*'), 1).alias('pct_emissao_cat'),
).show(1, False)

In [0]:
# ANÁLISE 1 — Perfil dos acidentes de trabalho no PR
# Universo: TODOS os acidentes de trabalho ocorridos no PR (não restrito a trânsito)
# Saída: laest.gold.cat_perfil_pr — formato longo (dimensão × categoria) com n e % por categoria
# % calculado dentro de cada dimensão (a soma por dimensão/ano é 100%)
from functools import reduce
from pyspark.sql import Window

pr = (spark.table(SILVER)
      .filter(F.col('uf_acidente') == '41')
      .withColumn('faixa_etaria',
          F.when(F.col('idade_anos') < 18, 'Menos de 18')
           .when(F.col('idade_anos') < 25, '18 a 24')
           .when(F.col('idade_anos') < 35, '25 a 34')
           .when(F.col('idade_anos') < 45, '35 a 44')
           .when(F.col('idade_anos') < 55, '45 a 54')
           .when(F.col('idade_anos') < 65, '55 a 64')
           .when(F.col('idade_anos').isNotNull(), '65 ou mais')
           .otherwise('Ignorado')))

# Dimensões do perfil (códigos preservados; tradução pendente no dicionário CAT)
DIMENSOES = {
    'ocupacao_cbo':          'Ocupação (CBO)',
    'sexo':                  'Sexo',
    'raca_cor':              'Raça/cor',
    'faixa_etaria':          'Faixa etária',
    'situacao_trabalho_cod': 'Situação no mercado de trabalho',
    'cat_emitida_cod':       'Emissão de CAT',
}

blocos = [
    pr.groupBy('ano', F.coalesce(F.col(col), F.lit('Ignorado')).alias('categoria'))
      .agg(F.count('*').alias('n'))
      .withColumn('dimensao', F.lit(nome))
    for col, nome in DIMENSOES.items()
]

perfil = reduce(lambda a, b: a.unionByName(b), blocos)
w = Window.partitionBy('ano', 'dimensao')
perfil = (perfil
    .withColumn('pct', F.round(100 * F.col('n') / F.sum('n').over(w), 3))
    .select('ano', 'dimensao', 'categoria', 'n', 'pct')
    .orderBy('dimensao', F.desc('n')))

perfil.write.mode('overwrite').saveAsTable('laest.gold.cat_perfil_pr')
spark.sql("""COMMENT ON TABLE laest.gold.cat_perfil_pr IS
'Perfil dos acidentes de trabalho ocorridos no PR (CAT/SINAN). Formato longo: dimensao x categoria com n e pct (pct calculado dentro de cada dimensao/ano). Universo: uf_acidente=41. Dimensoes: ocupacao CBO, sexo, raca/cor, faixa etaria, situacao no mercado de trabalho (codigo), emissao de CAT (codigo). Ignorados preservados. Fonte: laest.silver.cat_2025.'""")
print(f'{perfil.count():,} linhas gravadas em laest.gold.cat_perfil_pr')

In [0]:
# ANÁLISE 2 — Acidentes de trabalho NO TRÂNSITO no PR
# Universo: acidente ocorrido no PR com causa CID-10 V01-V99 (acidentes de transporte)
# Normalização: ponto final removido ('V23.' -> 'V23')
# Motocicleta: causas V20-V29
from pyspark.sql import Window

tr = (spark.table(SILVER)
      .filter(F.col('uf_acidente') == '41')
      .withColumn('causa', F.regexp_replace(F.col('causa_acidente_cid10'), r'\.$', ''))
      .filter(F.col('causa').rlike('^V')))

# (a) quantidade por ano + % envolvendo motocicleta
por_ano = (tr.groupBy('ano')
             .agg(F.count('*').alias('n_acidentes'),
                  F.count_if(F.col('causa').rlike('^V2')).alias('n_motocicleta'))
             .withColumn('pct_motocicleta',
                  F.round(100 * F.col('n_motocicleta') / F.col('n_acidentes'), 1))
             .orderBy('ano'))
por_ano.write.mode('overwrite').saveAsTable('laest.gold.cat_transito_pr_ano')
spark.sql("""COMMENT ON TABLE laest.gold.cat_transito_pr_ano IS
'Acidentes de trabalho no transito no PR, por ano (CAT/SINAN). Universo: uf_acidente=41 e causa CID-10 V01-V99 (ponto final normalizado). Colunas: n_acidentes, n_motocicleta (V20-V29) e pct_motocicleta. Fonte: laest.silver.cat_2025.'""")

# (b) por causa, com % dentro de cada ano (base para a evolução anual das principais causas)
# Descrições CID-10 de referência; códigos não mapeados ficam sem descrição
CID_TRANSITO = {
    'V01': 'Pedestre traumatizado em colisão com bicicleta',
    'V03': 'Pedestre traumatizado em colisão com automóvel, picape ou furgão',
    'V13': 'Pedestre traumatizado em colisão com veículo pesado ou ônibus',
    'V18': 'Ciclista traumatizado em outro acidente de transporte',
    'V22': 'Motociclista traumatizado em colisão com veículo de 2-3 rodas',
    'V23': 'Motociclista traumatizado em colisão com automóvel, picape ou furgão',
    'V24': 'Motociclista traumatizado em colisão com veículo pesado',
    'V28': 'Motociclista traumatizado em outro acidente de transporte (sem colisão)',
    'V29': 'Motociclista traumatizado em acidente de transporte não especificado',
    'V43': 'Ocupante de automóvel traumatizado em colisão com automóvel, picape ou furgão',
    'V89': 'Acidente de transporte com veículo a motor não especificado',
}

def descreve(colname, dicionario, padrao='Descrição não mapeada'):
    c = F.col(colname)
    pares = list(dicionario.items())
    expr = F.when(c == pares[0][0], F.lit(pares[0][1]))
    for chave, rotulo in pares[1:]:
        expr = expr.when(c == chave, F.lit(rotulo))
    return expr.otherwise(F.lit(padrao))

w_ano = Window.partitionBy('ano')
causas = (tr.groupBy('ano', 'causa')
            .agg(F.count('*').alias('n'),
                 F.max(F.col('causa').rlike('^V2').cast('int')).alias('motocicleta')))
causas = (causas
    .withColumn('pct_no_ano', F.round(100 * F.col('n') / F.sum('n').over(w_ano), 2))
    .withColumn('descricao', descreve('causa', CID_TRANSITO))
    .select('ano', 'causa', 'descricao', 'motocicleta', 'n', 'pct_no_ano')
    .orderBy('ano', F.desc('n')))
causas.write.mode('overwrite').saveAsTable('laest.gold.cat_transito_pr_causas')
spark.sql("""COMMENT ON TABLE laest.gold.cat_transito_pr_causas IS
'Acidentes de trabalho no transito no PR por ano e causa CID-10 (CAT/SINAN). Universo: uf_acidente=41 e causa V01-V99 normalizada. pct_no_ano calculado dentro de cada ano; flag motocicleta (V20-V29). Descricoes CID-10 de referencia. Fonte: laest.silver.cat_2025.'""")
print(f'{por_ano.count()} linhas em cat_transito_pr_ano | {causas.count()} linhas em cat_transito_pr_causas')

In [0]:
# ANÁLISE 3 — Lesões em acidentes de transporte no PR (top 10)
# Universo: o mesmo da análise 2 (`tr` e `descreve` vêm da célula anterior)
# % calculado sobre o total de acidentes de transporte no PR (universo da análise)
from pyspark.sql import Window

CID_LESAO_DESC = {
    'T07':  'Traumatismos que envolvem múltiplos locais do corpo',
    'S800': 'Contusão de joelho e perna (S80.0)',
    'T149': 'Lesão de região do corpo não especificada (T14.9)',
    'S400': 'Contusão de ombro e braço (S40.0)',
    'T00':  'Traumatismos superficiais que envolvem múltiplos locais do corpo',
    'S420': 'Fratura da clavícula (S42.0)',
    'T009': 'Traumatismo superficial múltiplo não especificado (T00.9)',
    'S202': 'Contusão do tórax (S20.2)',
    'S61':  'Ferimento do punho e da mão',
    'S009': 'Lesão superficial da cabeça, não especificada (S00.9)',
    'S80':  'Lesão superficial de joelho e perna',
    'S525': 'Fratura da extremidade distal do rádio (S52.5)',
    'T14':  'Lesão de região não especificada do corpo',
    'T008': 'Outros traumatismos superficiais múltiplos (T00.8)',
    'V29':  'Código de CAUSA gravado no campo de lesão (inconsistência da base)',
}

total_transito = tr.count()
w_rank = Window.orderBy(F.desc('n'))

lesoes = (tr.withColumn('lesao', F.regexp_replace(F.col('lesao_cid10'), r'\.$', ''))
            .groupBy(F.coalesce(F.col('lesao'), F.lit('Ignorado')).alias('lesao_cid10'))
            .agg(F.count('*').alias('n'))
            .orderBy(F.desc('n'))
            .limit(10))
lesoes = (lesoes
    .withColumn('ranking', F.row_number().over(w_rank))
    .withColumn('descricao', descreve('lesao_cid10', CID_LESAO_DESC))
    .withColumn('pct', F.round(100 * F.col('n') / total_transito, 1))
    .select('ranking', 'lesao_cid10', 'descricao', 'n', 'pct'))

lesoes.write.mode('overwrite').saveAsTable('laest.gold.cat_transito_pr_lesoes')
spark.sql("""COMMENT ON TABLE laest.gold.cat_transito_pr_lesoes IS
'Top 10 lesões (CID-10) em acidentes de trabalho no transito no PR (CAT/SINAN). Universo: uf_acidente=41 e causa V01-V99. pct calculado sobre o total de acidentes de transporte no PR. Descricoes CID-10 de referencia. Fonte: laest.silver.cat_2025.'""")
print(f'{lesoes.count()} lesões gravadas (de {total_transito:,} acidentes de transporte no PR)')

In [0]:
# VERIFICAÇÃO e dados prontos para gráficos
# 1) Checagem: soma dos % por dimensão deve ser ~100
spark.table('laest.gold.cat_perfil_pr') \
     .groupBy('ano', 'dimensao') \
     .agg(F.round(F.sum('pct'), 1).alias('soma_pct'), F.sum('n').alias('total')) \
     .orderBy('dimensao').show(10, False)

# 2) Dados estruturados para gráficos
print('— Análise 2: acidentes de trânsito por ano e % motocicleta —')
display(spark.table('laest.gold.cat_transito_pr_ano'))

print('— Análise 2: principais causas (top 10 do período) —')
spark.table('laest.gold.cat_transito_pr_causas') \
     .groupBy('causa', 'descricao', 'motocicleta') \
     .agg(F.sum('n').alias('n_total')) \
     .orderBy(F.desc('n_total')).show(10, False)

print('— Análise 3: top 10 lesões —')
display(spark.table('laest.gold.cat_transito_pr_lesoes'))

print('— Análise 1: top 5 categorias de cada dimensão do perfil —')
from pyspark.sql import Window as W
p = spark.table('laest.gold.cat_perfil_pr')
top5 = (p.withColumn('rk', F.row_number().over(W.partitionBy('dimensao').orderBy(F.desc('n'))))
          .filter(F.col('rk') <= 5)
          .orderBy('dimensao', 'rk'))
display(top5.drop('rk'))